# 01 — SCGM Text — analyse (lecture seule)

Notebook expérimental pour le pipeline SCGM-G sur embeddings texte fixes (sous-corpus BTP).


## 1. Objectif de l'expérience

- Source = BTP (segments de récits d'accidents).
- Labels macro = A0, A1, B, C (superclasses SCGM).
- SCGM apprend des ancres macro `mu_y` et des centres latents `mu_z`.
- Les composantes latentes servent à explorer des motifs intra-macro non observés.
- `pred_subtype` n'est pas un label expert : diagnostic exploratoire uniquement.


## 2. Imports et configuration

Notebook **lecture seule** : régler `OUTPUT_DIR` et `CHECKPOINT_PATH` dans la cellule paramètres. L'entraînement se fait via `scripts/train_scgm_text.py` ou `jobs/train_scgm_text.sh`.


In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import yaml

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# Parameters — lecture seule (entraînement via scripts/ ou jobs/)
OUTPUT_DIR = "resultats/scgm_text"
CHECKPOINT_PATH = None  # None → OUTPUT_DIR/checkpoints/best_model.pt
SKIP_EXPORT_IF_PRESENT = True  # True : export seulement si projected_embeddings.npy absent

DATA_CSV = "dataset/data_btp.csv"
EMB_CSV = "embeddings/Qwen3-Embedding-0.6B_btp.csv"
LABEL_COL = "pred_label"
PRED_OK_COL = "pred_ok"
GROUP_COL = "accident_id"
SEED = 42
VAL_RATIO = 0.1
BATCH_SIZE = 512  # export / évaluation

TSNE_SAMPLE_SIZE = 8000
DATAMAP_MAX_POINTS = 12000
RAW_EMBEDDING_UMAP_MAX_POINTS = 12000
DATAMAP_SEED = 42
DATAMAP_LABEL_MODE = "theme_summary"  # theme_summary | macro_z (theme_summary exige themes_by_z_openai.csv)
DATAMAP_SHOW_MACRO_CENTROIDS = True


In [ ]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "scgm_text").is_dir() and (candidate / "scripts").is_dir():
            return candidate
        if (candidate / "text" / "scgm_text").is_dir() and (candidate / "text" / "scripts").is_dir():
            return candidate / "text"
    raise FileNotFoundError("Impossible de localiser la racine text/ (scgm_text/ + scripts/).")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from scgm_text.dataset_text_embeddings import (
    ID2LABEL,
    LABEL2ID,
    TextEmbeddingDataset,
    split_by_group,
)
from scgm_text.utils_io import create_doc_id_if_missing, ensure_dir, get_dim_columns, load_json, save_json, set_seed

OUTPUT_PATH = Path(OUTPUT_DIR)
CHECKPOINTS_DIR = OUTPUT_PATH / "checkpoints"
CHECKPOINT_PATH = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else CHECKPOINTS_DIR / "best_model.pt"
EXPORTS_DIR = OUTPUT_PATH / "embeddings"
TOPICS_DIR = OUTPUT_PATH / "topics"
EVAL_DIR = OUTPUT_PATH / "metrics"
FIGURES_DIR = OUTPUT_PATH / "figures"
TABLES_DIR = OUTPUT_PATH / "tables"
for folder in [OUTPUT_PATH, CHECKPOINTS_DIR, EXPORTS_DIR, TOPICS_DIR, EVAL_DIR, FIGURES_DIR, TABLES_DIR]:
    ensure_dir(str(folder))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

run_config: dict = {}
_cfg_json = OUTPUT_PATH / "configs" / "config.json"
if _cfg_json.is_file():
    run_config = load_json(str(_cfg_json))


def save_fig(name: str) -> Path:
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path


def display_df_for_paper(df: pd.DataFrame, name: str) -> Path:
    path = TABLES_DIR / name
    df.to_csv(path, index=False)
    display(df)
    return path


def run_cli(cmd, stream=True):
    print(" ".join(cmd))
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if stream:
        completed = subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=False)
        if completed.returncode != 0:
            raise subprocess.CalledProcessError(completed.returncode, cmd)
        return completed
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode,
            cmd,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    return completed


def _read_training_logs(output_path: Path) -> pd.DataFrame:
    for candidate in (
        output_path / "metrics" / "train_log.csv",
        output_path / "logs.csv",
    ):
        if candidate.exists():
            return pd.read_csv(candidate)
    raise FileNotFoundError(f"Aucun journal trouvé sous {output_path}")


def show_training_progress(output_path=OUTPUT_PATH):
    try:
        logs_df = _read_training_logs(output_path)
    except FileNotFoundError as exc:
        print(exc)
        return
    display(logs_df.tail(5))
    if len(logs_df) == 0:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    loss_cols = [c for c in ["train_loss", "loss_macro", "loss_latent"] if c in logs_df.columns]
    if loss_cols:
        logs_df.plot(x="epoch", y=loss_cols, ax=axes[0])
    val_cols = [
        c
        for c in [
            "val_eta2_macro_balanced",
            "val_eta2_weighted",
            "val_macro_f1",
            "val_balanced_acc",
            "val_acc",
        ]
        if c in logs_df.columns
    ]
    if val_cols:
        logs_df.plot(x="epoch", y=val_cols, ax=axes[1])
    plt.show()


def require_scgm_artifacts(
    output_path: Path = OUTPUT_PATH,
    checkpoint_path: Path = CHECKPOINT_PATH,
) -> None:
    missing: list[str] = []
    ckpt = Path(checkpoint_path)
    if not ckpt.is_file():
        missing.append(str(ckpt))
    try:
        _read_training_logs(output_path)
    except FileNotFoundError:
        missing.append("metrics/train_log.csv ou logs.csv")
    if missing:
        raise FileNotFoundError(
            "Artefacts SCGM manquants — entraînez hors notebook, par ex.:\n"
            "  cd text/jobs && sbatch train_scgm_text.sh\n"
            "  python scripts/train_scgm_text.py --config configs/scgm_text_strict_fidelity.yaml "
            "--scgm_strict_mode --output_dir resultats/scgm_text\n"
            f"Manquant : {missing}"
        )


print(f"REPO_ROOT={REPO_ROOT}")
print(f"OUTPUT_DIR={OUTPUT_PATH}")
print(f"CHECKPOINT={CHECKPOINT_PATH}")
print(f"DEVICE={device}")
if run_config:
    print(
        "Run config:",
        run_config.get("fidelity_mode"),
        "| best epoch:",
        run_config.get("best_checkpoint_epoch"),
        "| metric:",
        run_config.get("best_checkpoint_metric"),
    )


## 3. Vérification environnement


In [ ]:
env_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "device": str(device),
    "cuda_available": torch.cuda.is_available(),
    "cpu_count": os.cpu_count(),
}
if torch.cuda.is_available():
    env_info["gpu_name"] = torch.cuda.get_device_name(0)
    env_info["gpu_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
pd.Series(env_info)


## 4. Chargement et inspection des données brutes


In [ ]:
raw_df = pd.read_csv(DATA_CSV)
print("shape:", raw_df.shape)
display(raw_df.head())
print("colonnes:", list(raw_df.columns))
print("accidents distincts:", raw_df[GROUP_COL].nunique())
print("pred_ok True/False:", raw_df[PRED_OK_COL].astype(str).str.lower().value_counts().to_dict())
print("NaN pred_label:", int(raw_df[LABEL_COL].isna().sum()))
print("distribution pred_label (brute):")
display(raw_df[LABEL_COL].value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
raw_df[LABEL_COL].value_counts(dropna=False).plot(kind="bar", ax=axes[0], title="pred_label (brut)")
if "word_count" in raw_df.columns:
    raw_df["word_count"].hist(ax=axes[1], bins=40)
    axes[1].set_title("word_count")
if "pred_confidence" in raw_df.columns:
    raw_df["pred_confidence"].dropna().hist(ax=axes[2], bins=40)
    axes[2].set_title("pred_confidence")
save_fig("01_raw_distributions.png")


## 5. Chargement et inspection des embeddings


In [ ]:
emb_preview = pd.read_csv(EMB_CSV, nrows=5)
display(emb_preview.iloc[:, :8])
header = pd.read_csv(EMB_CSV, nrows=0)
dim_cols = get_dim_columns(header)
print("input_dim:", len(dim_cols))

doc_ids = pd.read_csv(EMB_CSV, usecols=["doc_id"])["doc_id"]
print("embedding rows:", len(doc_ids))
print("doc_id min/max:", int(doc_ids.min()), int(doc_ids.max()))
print("doc_id unique:", doc_ids.nunique() == len(doc_ids))

raw_with_doc = create_doc_id_if_missing(raw_df)
assert len(raw_with_doc) == 42309
assert doc_ids.min() == 1 and doc_ids.max() == len(raw_with_doc)


## 6. Dataset filtré SCGM


In [ ]:
dataset = TextEmbeddingDataset(
    data_csv=DATA_CSV,
    emb_csv=EMB_CSV,
    label_col=LABEL_COL,
    pred_ok_col=PRED_OK_COL,
    group_col=GROUP_COL,
)
meta = dataset.get_metadata_df()
print("lignes filtrées:", len(dataset))
print("distribution macro:", dataset.get_label_distribution())
print("accidents distincts:", meta[GROUP_COL].nunique())
print("input_dim:", dataset.get_input_dim())

examples = (
    meta.groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: g.sample(min(2, len(g)), random_state=SEED))[["accident_id", "fact_id", LABEL_COL, "sentence"]]
)
display(examples)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
meta[LABEL_COL].value_counts().reindex(["A0", "A1", "B", "C"]).plot(kind="bar", ax=axes[0], title="Macro filtrées")
if "word_count" in meta.columns:
    meta.boxplot(column="word_count", by=LABEL_COL, ax=axes[1])
    axes[1].set_title("word_count par macro")
if "pred_confidence" in meta.columns:
    for label in ["A0", "A1", "B", "C"]:
        subset = meta.loc[meta[LABEL_COL] == label, "pred_confidence"].dropna()
        axes[2].hist(subset, bins=30, alpha=0.5, label=label)
    axes[2].legend()
    axes[2].set_title("pred_confidence par macro")
save_fig("02_filtered_distributions.png")

## 7. Split train/validation par accident_id


In [ ]:
train_idx, val_idx = split_by_group(dataset, val_ratio=VAL_RATIO, seed=SEED)
train_meta = meta.iloc[train_idx].copy()
val_meta = meta.iloc[val_idx].copy()
print("train units:", len(train_idx), "| val units:", len(val_idx))
print("train accidents:", train_meta[GROUP_COL].nunique(), "| val accidents:", val_meta[GROUP_COL].nunique())
intersection = set(train_meta[GROUP_COL]).intersection(set(val_meta[GROUP_COL]))
print("intersection accident_id train/val:", len(intersection))
print("train label dist:", train_meta[LABEL_COL].value_counts().to_dict())
print("val label dist:", val_meta[LABEL_COL].value_counts().to_dict())


In [ ]:
split_df = pd.DataFrame({
    "macro": ["A0", "A1", "B", "C"],
    "train": [int((train_meta[LABEL_COL] == m).sum()) for m in ["A0", "A1", "B", "C"]],
    "val": [int((val_meta[LABEL_COL] == m).sum()) for m in ["A0", "A1", "B", "C"]],
}).set_index("macro")
split_df.plot(kind="bar", title="Distribution macro train vs val")
plt.xlabel("macro")
save_fig("03_train_val_macro.png")
display_df_for_paper(split_df.reset_index(), "split_train_val.csv")


## 8. Résultats d'entraînement (lecture seule)

**Pas d'entraînement dans ce notebook.** Produire d'abord les artefacts sous `OUTPUT_DIR` :

```bash
cd text/jobs && sbatch train_scgm_text.sh
# ou
python scripts/train_scgm_text.py \
  --config configs/scgm_text_strict_fidelity.yaml \
  --scgm_strict_mode \
  --output_dir resultats/scgm_text
```

Puis export (si `embeddings/projected_embeddings.npy` absent) :

```bash
python scripts/export_scgm_text_outputs.py \
  --checkpoint resultats/scgm_text/checkpoints/best_model.pt \
  --output_dir resultats/scgm_text
```

Attendu : `checkpoints/best_model.pt`, `metrics/train_log.csv` (ou `logs.csv`), puis exports sous `embeddings/` et `topics/`.

Libellés OpenAI pour la carte DataMap : `bash jobs/enrich_scgm_themes_openai.sh` sur le **login** (pas dans ce notebook).


In [ ]:
require_scgm_artifacts()

if run_config:
    display(
        pd.Series(
            {
                "fidelity_mode": run_config.get("fidelity_mode"),
                "best_checkpoint_epoch": run_config.get("best_checkpoint_epoch"),
                "best_checkpoint_metric": run_config.get("best_checkpoint_metric"),
                "best_checkpoint_score": run_config.get("best_checkpoint_score"),
                "output_dir": run_config.get("output_dir", str(OUTPUT_PATH)),
            }
        )
    )

show_training_progress()


## 9. Courbes d'entraînement


In [ ]:
logs = _read_training_logs(OUTPUT_PATH)
display(logs.tail())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
loss_cols = [c for c in ["train_loss", "loss_macro", "loss_latent"] if c in logs.columns]
if loss_cols:
    logs.plot(x="epoch", y=loss_cols, ax=axes[0, 0])
val_cols = [
    c
    for c in [
        "val_eta2_macro_balanced",
        "val_eta2_weighted",
        "val_acc",
        "val_macro_f1",
        "val_balanced_acc",
    ]
    if c in logs.columns
]
if val_cols:
    logs.plot(x="epoch", y=val_cols, ax=axes[0, 1])
    axes[0, 1].set_title("Validation (η² ou classif)")
geom_cols = [c for c in ["rankme_global", "c1_global", "c10_global"] if c in logs.columns]
if geom_cols:
    logs.plot(x="epoch", y=geom_cols, ax=axes[1, 0], marker="o", markersize=3)
    axes[1, 0].set_title("RankMe / C1 / C10 (global)")
axes[1, 1].axis("off")
save_fig("04_training_curves.png")
display_df_for_paper(logs, "training_logs.csv")

from scgm_text.notebook_viz import plot_training_geometry_curves

plot_training_geometry_curves(logs, save_fig=save_fig)


## 10. Meilleur checkpoint


In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
config = load_json(str(OUTPUT_PATH / "configs" / "config.json"))
summary_ckpt = {
    "input_dim": checkpoint.get("input_dim"),
    "label2id": checkpoint.get("label2id"),
    "train_size": len(checkpoint.get("train_idx", [])),
    "val_size": len(checkpoint.get("val_idx", [])),
    "args": checkpoint.get("args", {}),
}
display(pd.json_normalize(summary_ckpt["args"]))
pd.Series({
    "input_dim": summary_ckpt["input_dim"],
    "train_size": summary_ckpt["train_size"],
    "val_size": summary_ckpt["val_size"],
})


## 11. Export des sorties


In [ ]:
def _verify_export_layout() -> None:
    required = {
        "projected_embeddings.npy": EXPORTS_DIR / "projected_embeddings.npy",
        "metadata_with_predictions.csv": EXPORTS_DIR / "metadata_with_predictions.csv",
        "themes_by_z.csv": TOPICS_DIR / "themes_by_z.csv",
    }
    missing = {k: str(v) for k, v in required.items() if not v.is_file()}
    if not missing:
        return
    nested = EXPORTS_DIR / "embeddings" / "projected_embeddings.npy"
    if nested.is_file():
        raise FileNotFoundError(
            "Export écrit sous embeddings/embeddings/ (mauvais --output_dir). "
            f"Relancez avec --output_dir {OUTPUT_PATH} (racine du run, pas le sous-dossier embeddings/)."
        )
    raise FileNotFoundError(f"Export incomplet — fichiers manquants : {missing}")


_proj = EXPORTS_DIR / "projected_embeddings.npy"
if SKIP_EXPORT_IF_PRESENT and _proj.is_file():
    print(f"Export ignoré (déjà présent) : {_proj}")
else:
    export_cmd = [
        sys.executable,
        "scripts/export_scgm_text_outputs.py",
        "--data_csv", DATA_CSV,
        "--emb_csv", EMB_CSV,
        "--checkpoint", str(CHECKPOINT_PATH),
        "--output_dir", str(OUTPUT_PATH),
        "--label_col", LABEL_COL,
        "--pred_ok_col", PRED_OK_COL,
        "--group_col", GROUP_COL,
        "--batch_size", str(BATCH_SIZE),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    run_cli(export_cmd)
    for sub in ("embeddings", "topics", "assignments"):
        folder = OUTPUT_PATH / sub
        if folder.is_dir():
            names = sorted(p.name for p in folder.iterdir() if p.is_file())
            print(f"{sub}/ ({len(names)} fichiers):", names[:12], "..." if len(names) > 12 else "")

_verify_export_layout()
print("Export OK —", _proj)


## 11 bis — Carte 2D des segments (UMAP + DataMapPlot)

Sous-échantillon (`DATAMAP_MAX_POINTS`). UMAP sur `projected_embeddings.npy`.

Libellés OpenAI (`theme_summary`) : produits **hors notebook** avec
`bash jobs/enrich_scgm_themes_openai.sh` sur le **login** (accès Internet), après export SCGM.
Fichier attendu : `topics/themes_by_z_openai.csv`. Sinon `DATAMAP_LABEL_MODE = "macro_z"`.

`DATAMAP_SHOW_MACRO_CENTROIDS` : marqueurs `P` = moyenne UMAP par macro (A0–C).

- Statique : `datamap_segments.png`
- Interactif Plotly : `datamap_segments_interactive.html`


In [ ]:
from umap import UMAP

from scgm_text.notebook_viz import (
    display_plotly_html,
    macro_umap_centroids,
    plot_umap_datamap_static,
    plot_umap_plotly,
    resolve_datamap_labels,
)

emb_path = EXPORTS_DIR / "projected_embeddings.npy"
meta_path = EXPORTS_DIR / "metadata_with_predictions.csv"
themes_openai_path = TOPICS_DIR / "themes_by_z_openai.csv"

if not emb_path.exists() or not meta_path.exists():
    print("Embeddings projetés ou métadonnées manquants ; exécuter l'export.")
else:
    projected_all = np.load(emb_path)
    meta_all = pd.read_csv(meta_path)
    if len(meta_all) != len(projected_all):
        raise ValueError(f"Alignement meta/embeddings : {len(meta_all)} vs {len(projected_all)}")
    n = min(DATAMAP_MAX_POINTS, len(projected_all))
    rng = np.random.default_rng(DATAMAP_SEED)
    idx = rng.choice(len(projected_all), size=n, replace=False)
    X = projected_all[idx]
    lab = meta_all.iloc[idx].copy()

    labels, label_kind = resolve_datamap_labels(
        lab,
        label_col=LABEL_COL,
        label_mode=DATAMAP_LABEL_MODE,
        themes_openai_path=themes_openai_path,
    )
    if label_kind == "theme_summary":
        _to = pd.read_csv(themes_openai_path)
        _z2s = dict(zip(_to["z_id"].astype(int), _to["theme_summary"].astype(str)))
        lab["hover_theme"] = lab["z_hat"].map(lambda z: _z2s.get(int(z), f"z={int(z)}"))
    else:
        lab["hover_theme"] = lab[LABEL_COL].astype(str) + "|z=" + lab["z_hat"].astype(str)

    reducer = UMAP(
        n_components=2,
        random_state=DATAMAP_SEED,
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
    )
    coords = reducer.fit_transform(X)

    centroids = None
    if DATAMAP_SHOW_MACRO_CENTROIDS:
        centroids = macro_umap_centroids(coords, lab[LABEL_COL].astype(str).to_numpy())

    fig, _ax = plot_umap_datamap_static(
        coords,
        labels,
        title="Segments BTP — embedding SCGM (normalisé)",
        label_font_size=8,
        macro_centroids=centroids,
    )
    out_png = FIGURES_DIR / "datamap_segments.png"
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(out_png)

    fig_pl = plot_umap_plotly(
        coords,
        lab,
        label_col=LABEL_COL,
        hover_label="hover_theme",
        title="UMAP segments BTP (interactif)",
        out_html=FIGURES_DIR / "datamap_segments_interactive.html",
    )
    display_plotly_html(FIGURES_DIR / "datamap_segments_interactive.html")


## 12. Visualisation des embeddings projetés

PCA + t-SNE sur un sous-échantillon (`TSNE_SAMPLE_SIZE`). Couleur = macro (`pred_label`).

- Statique : `FIGURES_DIR/05_projection_macro.png`
- Interactif Plotly : `05_projection_pca_interactive.html`, `05_projection_tsne_interactive.html`


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scgm_text.notebook_viz import (
    display_plotly_html,
    plot_projection_matplotlib,
    plot_projection_plotly,
    sample_projection_indices,
)

projected = np.load(EXPORTS_DIR / "projected_embeddings.npy")
meta_pred = pd.read_csv(EXPORTS_DIR / "metadata_with_predictions.csv")

idx = sample_projection_indices(meta_pred, LABEL_COL, max_points=TSNE_SAMPLE_SIZE, seed=SEED)
sample_df = meta_pred.loc[idx]
sample_x = projected[idx]

pca = PCA(n_components=2, random_state=SEED)
pca_xy = pca.fit_transform(sample_x)
tsne = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto")
tsne_xy = tsne.fit_transform(sample_x)

plot_projection_matplotlib(pca_xy, tsne_xy, sample_df, LABEL_COL, save_fig=save_fig)
_, pca_html = plot_projection_plotly(pca_xy, tsne_xy, sample_df, LABEL_COL, figures_dir=FIGURES_DIR)
print(pca_html)
display_plotly_html(FIGURES_DIR / "05_projection_pca_interactive.html")
display_plotly_html(FIGURES_DIR / "05_projection_tsne_interactive.html")


## 13. Analyse des centres macro mu_y


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

mu_y = np.load(EXPORTS_DIR / "mu_y.npy")
macro_names = ["A0", "A1", "B", "C"]
cos_y = cosine_similarity(mu_y)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cos_y, annot=True, fmt=".2f", xticklabels=macro_names, yticklabels=macro_names, ax=ax)
ax.set_title("Similarité cosinus entre mu_y")
save_fig("06_mu_y_cosine.png")

norms = np.linalg.norm(mu_y, axis=1)
macro_table = pd.DataFrame({"macro": macro_names, "norm": norms})
display_df_for_paper(macro_table, "mu_y_norms.csv")


## 14. Analyse des centres latents mu_z


In [ ]:
mu_z = np.load(EXPORTS_DIR / "mu_z.npy")
z_counts = meta_pred["z_hat"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
z_counts.plot(kind="bar", ax=axes[0], title="Effectifs par z_hat")
pd.crosstab(meta_pred["z_hat"], meta_pred[LABEL_COL]).plot(kind="bar", stacked=True, ax=axes[1], legend=True)
axes[1].set_title("z_hat x pred_label")
save_fig("07_mu_z_usage.png")

cos_z = cosine_similarity(mu_z)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cos_z, cmap="vlag", center=0, ax=ax)
ax.set_title("Similarité cosinus mu_z")
save_fig("08_mu_z_cosine.png")


## 15. Thèmes par composante z


In [ ]:
themes_z = pd.read_csv(EXPORTS_DIR / "themes_by_z.csv")
themes_macro = pd.read_csv(EXPORTS_DIR / "themes_by_macro_z.csv")
display(themes_macro)
top_components = themes_z.sort_values("n_units", ascending=False).head(15)
display(top_components[["z_id", "dominant_macro", "n_units", "top_words"]])
display_df_for_paper(top_components, "top_themes_by_z.csv")


## 16. Métriques géométriques


### Carte 2D — embedding brut (PCA + t-SNE statiques, couleur = macro)

Sous-échantillon (`RAW_EMBEDDING_UMAP_MAX_POINTS`) sur les vecteurs **encodeur** (`EMB_CSV` / `raw_embeddings.npy`), colorés par `pred_label` (A0–C). Figure statique : `09_raw_embedding_pca_tsne.png`.


In [ ]:
eval_cmd = [
    sys.executable,
    "scripts/evaluate_scgm_text.py",
    "--exports_dir", str(EXPORTS_DIR),
    "--output_dir", str(EVAL_DIR),
    "--label_col", LABEL_COL,
    "--emb_csv", EMB_CSV,
]
run_cli(eval_cmd)

import importlib
import scgm_text.notebook_viz as _notebook_viz
importlib.reload(_notebook_viz)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scgm_text.notebook_viz import (
    display_plotly_html,
    plot_evaluation_geometry_dashboard,
    plot_projection_matplotlib,
    sample_projection_indices,
)

metrics_table = pd.read_csv(EVAL_DIR / "metrics_table.csv")
display(metrics_table)

html_paths = plot_evaluation_geometry_dashboard(
    metrics_table,
    figures_dir=FIGURES_DIR,
    save_fig=save_fig,
)
for p in html_paths:
    print(p)
display_plotly_html(FIGURES_DIR / "09_eta2_macro_interactive.html")
display_plotly_html(FIGURES_DIR / "09_rankme_c1_c10_global_interactive.html")

# --- PCA / t-SNE 2D embedding brut (macro, statique) ---
meta_eval = pd.read_csv(EXPORTS_DIR / "metadata_with_predictions.csv")
raw_npy = EXPORTS_DIR / "raw_embeddings.npy"
if raw_npy.is_file():
    raw_emb = np.load(raw_npy)
elif Path(EMB_CSV).is_file():
    from scgm_text.dataset_text_embeddings import merge_metadata_with_embeddings
    slim = meta_eval.drop(columns=[c for c in meta_eval.columns if c.startswith("dim_")], errors="ignore")
    merged, dim_cols = merge_metadata_with_embeddings(slim, EMB_CSV)
    raw_emb = merged[dim_cols].to_numpy(dtype=np.float64)
else:
    raise FileNotFoundError("raw_embeddings.npy ou EMB_CSV requis pour la carte embedding brut.")

idx_raw = sample_projection_indices(
    meta_eval, LABEL_COL, max_points=RAW_EMBEDDING_UMAP_MAX_POINTS, seed=SEED
)
sample_raw_df = meta_eval.loc[idx_raw]
sample_raw_x = raw_emb[idx_raw]

pca_raw = PCA(n_components=2, random_state=SEED)
pca_raw_xy = pca_raw.fit_transform(sample_raw_x)
tsne_raw = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto")
tsne_raw_xy = tsne_raw.fit_transform(sample_raw_x)

p_raw = plot_projection_matplotlib(
    pca_raw_xy,
    tsne_raw_xy,
    sample_raw_df,
    LABEL_COL,
    save_fig=save_fig,
    png_name="09_raw_embedding_pca_tsne.png",
    pca_title="PCA 2D — embedding brut (macro)",
    tsne_title="t-SNE 2D — embedding brut (macro)",
)
print(p_raw)


## 17. Tableaux prêts pour papier / thèse


In [ ]:
display_df_for_paper(metrics_table, "paper_metrics_summary.csv")
display_df_for_paper(themes_macro, "paper_themes_by_macro.csv")

notebook_summary = {
    "output_dir": str(OUTPUT_PATH),
    "exports_dir": str(EXPORTS_DIR),
    "evaluation_dir": str(EVAL_DIR),
    "figures_dir": str(FIGURES_DIR),
    "tables_dir": str(TABLES_DIR),
    "best_checkpoint_epoch": run_config.get("best_checkpoint_epoch"),
    "device": str(device),
    "figure_files": sorted(p.name for p in FIGURES_DIR.glob("*.png")),
    "table_files": sorted(p.name for p in TABLES_DIR.glob("*.csv")),
}
save_json(notebook_summary, OUTPUT_PATH / "notebook_summary.json")
notebook_summary
